# 🚗 Treinamento YOLO11m para Detecção de Tráfego
**Google Colab - GPU Gratuita**

Este notebook treina um modelo YOLO11m para detecção de veículos em vídeos de tráfego rodoviário.

## O que este notebook faz:
1. ✅ Instala dependências (ultralytics, opencv, etc.)
2. ✅ Baixa o modelo pré-treinado yolo11m.pt
3. ✅ Treina no seu dataset de vídeos de tráfego
4. ✅ Valida o modelo treinado
5. ✅ Exporta o modelo final (.pt) para download

---
**⚠️ Importante:** Ative a GPU no Colab antes de começar:
`Menu → Runtime → Change runtime type → GPU → Save`

**Tempo estimado:** 30-60 minutos para 50 épocas.

In [ ]:
# ============================================================================
# PASSO 1: Instalar dependências
# ============================================================================
!pip install -q ultralytics opencv-python numpy

print("✅ Dependências instaladas com sucesso!")

In [ ]:
# ============================================================================
# PASSO 2: Baixar o modelo pré-treinado yolo11m.pt
# ============================================================================
import os
from pathlib import Path

os.makedirs('models', exist_ok=True)

model_path = 'models/yolo11m.pt'
if not os.path.exists(model_path):
    print('📥 Baixando modelo yolo11m.pt...')
    !wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt -O models/yolo11m.pt
    print('✅ Modelo baixado!')
else:
    print('✅ Modelo já existe localmente.')

print(f"Tamanho do modelo: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

In [ ]:
# ============================================================================
# PASSO 3: Preparar o dataset
# ============================================================================
!pip install -q gdown

import os
from pathlib import Path

os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

print('📁 Pastas criadas:')
print('  dataset/images/train/')
print('  dataset/images/val/')
print('  dataset/labels/train/')
print('  dataset/labels/val/')

print('\n📋 Agora carregue seus vídeos e anotações para estas pastas:')
print('   - Coloque as imagens/frames em images/train/ e images/val/')
print('   - Coloque as anotações .txt em labels/train/ e labels/val/')
print('   \n   Ou faça upload direto pelo ícone de pasta no painel esquerdo do Colab.')

In [ ]:
# ============================================================================
# PASSO 4: Criar arquivo data.yaml
# ============================================================================
yaml_content = '''train: dataset/images/train
val: dataset/images/val
nc: 6
names:
  - person
  - car
  - motorcycle
  - bus
  - truck
  - articulated_truck'''

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ Arquivo data.yaml criado:')
print(yaml_content)

In [ ]:
# ============================================================================
# PASSO 5: Treinar o modelo
# ============================================================================
from ultralytics import YOLO

# Carregar modelo pré-treinado
model = YOLO('models/yolo11m.pt')

# Treinar
results = model.train(
    data='data.yaml',
    epochs=50,          # Número de épocas (aumente para 100+ se tiver tempo)
    imgsz=640,          # Tamanho da imagem
    batch=16,           # Tamanho do lote (reduza se der erro de memória)
    device='0',         # Usar GPU
    project='runs/train',
    name='traffic_analysis',
    exist_ok=True,
    patience=20,        # Parar se não melhorar por 20 épocas
    save=True,
    verbose=True
)

print('\n✅ Treinamento concluído!')
print(f"📁 Modelo salvo em: runs/train/traffic_analysis/weights/best.pt")

In [ ]:
# ============================================================================
# PASSO 6: Validar o modelo treinado
# ============================================================================
from ultralytics import YOLO

model = YOLO('runs/train/traffic_analysis/weights/best.pt')

metrics = model.val(
    data='data.yaml',
    imgsz=640,
    batch=16,
    device='0'
)

print('\n✅ Validação concluída!')
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precisão: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

In [ ]:
# ============================================================================
# PASSO 7: Baixar o modelo treinado
# ============================================================================
from google.colab import files

model_path = 'runs/train/traffic_analysis/weights/best.pt'

if os.path.exists(model_path):
    print(f"📥 Baixando modelo: {model_path}")
    files.download(model_path)
    print('✅ Download iniciado!')
else:
    print('❌ Modelo não encontrado. Verifique se o treinamento foi concluído.')

# 📋 Como usar o modelo treinado no seu projeto

1. Baixe o arquivo `best.pt` que foi gerado acima
2. Substitua o arquivo `models/yolo11m.pt` no seu projeto pelo novo modelo
3. Execute normalmente:
   ```bat
   run.bat --mode vehicles --source assets/seu_video.mp4 --no-show
   ```

---
## 📚 Dicas para melhorar o resultado:
- Use pelo menos 100-200 imagens anotadas para treino
-> Variedade de ângulos, iluminação e clima
- Anote apenas as classes que você precisa (car, truck, bus, motorcycle, person)
- Use `--conf 0.25` para detecção padrão ou `--conf 0.40` para detecção mais conservadora

---
🏁 **Fim do notebook**